# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** [Tuliskan Nama Anda]  
**NIM:** [Tuliskan NIM Anda]  
**Username GitHub:** [Tuliskan Username GitHub Anda]  

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.


### Ruang lingkup dan keputusan analisis
Analisis menggunakan hasil Milestone 2, bukan data mentah. TMDB adalah katalog film global; hasil tidak mewakili seluruh industri film Indonesia. Kesesuaian pemilihan dataset dengan syarat “Dataset Indonesia” perlu dikonfirmasi kepada dosen. Identitas mahasiswa di atas diisi oleh pemilik tugas.

Analisis temporal dibatasi **2000–2023** dan status **Released**, memakai tahun kalender penuh yang konservatif terhadap versi dataset. Tanggal sesudah 2023, tanggal kosong/tidak valid, dan status lain dilaporkan sebagai audit. Tahun rilis bukan tahun pencatatan; perubahan jumlah film dapat dipengaruhi kelengkapan katalog. Rating dianalisis hanya bila `vote_count >= 100` dan rating 0–10. Nilai anggaran/pendapatan nol atau negatif tidak digunakan dalam analisis finansial, karena nol dapat berarti tidak tersedia. Popularity tidak dianalisis karena telah di-clip pada Milestone 2.

Sumber template: https://github.com/UMM-GURU/tugas1-abd-2026/blob/main/notebooks/03_eda_and_insights.ipynb


## Bagian 1: Inisialisasi dan Memuat Data Bersih

In [1]:
from pathlib import Path
import polars as pl
import duckdb
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / 'notebooks').is_dir():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/processed/TMDB_movie_dataset_clean.parquet'
OUT = ROOT / 'output/figures'
TABLES = ROOT / 'output/tables'
OUT.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError('Jalankan notebooks/02_data_cleaning.ipynb terlebih dahulu.')
START, END, MIN_VOTES = 2000, 2023, 100
pio.renderers.default = 'plotly_mimetype'
px.defaults.template = 'plotly_white'
lf = pl.scan_parquet(DATA_PATH).with_columns(
    pl.col('release_date').str.to_date('%Y-%m-%d', strict=False).alias('tanggal')
)
audit = lf.select(
    pl.len().alias('total_baris'), pl.col('id').n_unique().alias('id_unik'),
    pl.col('tanggal').is_null().sum().alias('tanggal_kosong_atau_invalid'),
    (pl.col('tanggal').dt.year() > END).sum().alias('tanggal_setelah_2023'),
    (pl.col('status') != 'Released').fill_null(True).sum().alias('status_bukan_released'),
    (pl.col('revenue') < 0).sum().alias('revenue_negatif'),
    (pl.col('runtime') < 0).sum().alias('runtime_negatif')
).collect()
assert audit['total_baris'][0] == audit['id_unik'][0], 'ID belum unik'
df = lf.filter(
    pl.col('tanggal').dt.year().is_between(START, END) & (pl.col('status') == 'Released')
).select('id', 'title', 'tanggal', 'vote_average', 'vote_count', 'budget', 'revenue',
         'runtime', 'genres', 'original_language', 'production_countries').with_columns(
    pl.col('tanggal').dt.year().alias('tahun'),
    pl.col('tanggal').dt.month().alias('bulan')
).collect(engine='streaming')
rated = df.filter((pl.col('vote_count') >= MIN_VOTES) & pl.col('vote_average').is_between(0,10))
financial = rated.filter((pl.col('budget') > 0) & (pl.col('revenue') > 0)).with_columns(
    (pl.col('revenue') / pl.col('budget')).alias('rasio_pendapatan_anggaran')
)
display(audit)
print(f'Kohort temporal: {df.height:,}; rating: {rated.height:,}; finansial: {financial.height:,}')
audit.write_csv(TABLES / 'audit_eda.csv')


Kohort temporal: 614,677; rating: 12,677; finansial: 4,689


total_baris,id_unik,tanggal_kosong_atau_invalid,tanggal_setelah_2023,status_bukan_released,revenue_negatif,runtime_negatif
u32,u32,u32,u32,u32,u32,u32
1163770,1163770,198566,32450,27829,1,1


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)
Agregasi rilis per tahun memakai Polars `group_by_dynamic`. Kalender lengkap menjamin rolling mean 3 baris berarti 3 tahun. DuckDB `LAG ... OVER` menghitung perubahan tahunan; tahun pertama tidak memiliki pembanding.

In [2]:
annual_raw = df.sort('tanggal').group_by_dynamic('tanggal', every='1y').agg(pl.len().alias('jumlah_film')).with_columns(pl.col('tanggal').dt.year().alias('tahun'))
annual = pl.DataFrame({'tahun': list(range(START, END+1))}).join(
    annual_raw.select('tahun','jumlah_film'), on='tahun', how='left'
).with_columns(pl.col('jumlah_film').fill_null(0).cast(pl.Int64)).sort('tahun').with_columns(
    pl.col('jumlah_film').rolling_mean(window_size=3, min_samples=3).alias('rata_rata_3_tahun')
)
con = duckdb.connect()
con.register('annual', annual.to_arrow())
annual = con.execute("""
SELECT *, jumlah_film - LAG(jumlah_film) OVER (ORDER BY tahun) AS perubahan,
100.0 * (jumlah_film - LAG(jumlah_film) OVER (ORDER BY tahun)) /
NULLIF(LAG(jumlah_film) OVER (ORDER BY tahun), 0) AS pertumbuhan_persen
FROM annual ORDER BY tahun
""").pl()
con.close()
assert annual.height == END-START+1 and annual['jumlah_film'].sum() == df.height
display(annual)
annual.write_csv(TABLES / 'tren_tahunan.csv')
figures = []
def save_show(fig, name):
    fig.update_layout(margin=dict(t=80, b=70), title_x=0.02)
    fig.write_html(OUT / f'{name}.html', include_plotlyjs=True, full_html=True)
    figures.append((name, fig))
    fig.show()


tahun,jumlah_film,rata_rata_3_tahun,perubahan,pertumbuhan_persen
i64,i64,f64,i64,f64
2000,8735,null,null,null
2001,9835,null,1100,12.593017
2002,10962,9844.0,1127,11.459075
2003,12331,11042.666667,1369,12.488597
2004,13812,12368.333333,1481,12.01038
…,…,…,…,…
2019,41552,38137.333333,3900,10.358016
2020,40724,39976.0,-828,-1.992684
2021,43967,42081.0,3243,7.963363


## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Bangun minimal 6 visualisasi interaktif yang mencakup:
- Visualisasi 1: Tren deret waktu (Line chart dengan range slider)
- Visualisasi 2: Distribusi variabel utama (Box plot / Histogram / Violin plot)
- Visualisasi 3: Korelasi antar-variabel numerik (Heatmap / Scatter plot sampling)
- Visualisasi 4: Komposisi kategori utama (Bar chart / Treemap)
- Visualisasi 5: Analisis siklus musiman / perbandingan kelompok
- Visualisasi 6: Pola spasial atau korelasi multi-dimensi

In [3]:
fig1 = go.Figure()
fig1.add_scatter(x=annual['tahun'].to_list(), y=annual['jumlah_film'].to_list(), name='Jumlah film', mode='lines+markers')
fig1.add_scatter(x=annual['tahun'].to_list(), y=annual['rata_rata_3_tahun'].to_list(), name='Rata-rata bergerak 3 tahun')
fig1.update_layout(title='1. Tren jumlah film Released, 2000–2023', xaxis_title='Tahun rilis', yaxis_title='Jumlah film', xaxis_rangeslider_visible=True)
save_show(fig1, '01_tren_tahunan')


### Visualisasi 2: Distribusi rating
Histogram mencakup seluruh kohort dengan minimal 100 vote, tanpa sampling. Rating tinggi pada film minim vote tidak langsung dianggap performa yang kuat.

In [4]:
fig2 = px.histogram(rated.to_dict(as_series=False), x='vote_average', nbins=40, title=f'2. Distribusi rating (n={rated.height:,}; minimal {MIN_VOTES} vote)', labels={'vote_average':'Rating rata-rata'})
fig2.update_layout(yaxis_title='Jumlah film')
save_show(fig2, '02_distribusi_rating')


### Visualisasi 3: Korelasi numerik
Pearson pada kohort finansial dengan minimal 100 vote. Transformasi log10 untuk anggaran, pendapatan, dan vote mengurangi dominasi skala ekstrem; korelasi bukan bukti sebab-akibat. Runtime dibatasi >0 tanpa membuang film dari kohort utama.

In [5]:
corr_data = financial.filter(pl.col('runtime') > 0).select(
    pl.col('budget').log10().alias('log10 anggaran'),
    pl.col('revenue').log10().alias('log10 pendapatan'),
    pl.col('vote_count').log10().alias('log10 vote'),
    pl.col('vote_average').alias('rating'), pl.col('runtime').alias('durasi')
)
corr = corr_data.corr()
fig3 = px.imshow(corr.to_numpy(), x=corr.columns, y=corr.columns, zmin=-1, zmax=1, color_continuous_scale='RdBu_r', text_auto='.2f', title=f'3. Korelasi Pearson pada kohort finansial (n={corr_data.height:,})')
save_show(fig3, '03_korelasi_numerik')
corr.write_csv(TABLES / 'korelasi.csv')


### Visualisasi 4: Komposisi genre
Genre dipisah berdasarkan koma dan dihitung per film unik. Satu film dapat memiliki beberapa genre, sehingga jumlah antargenre tidak boleh dijumlahkan sebagai total film.

In [6]:
genres = df.select('id', pl.col('genres').str.split(',').alias('genre')).explode('genre').with_columns(pl.col('genre').str.strip_chars()).filter(pl.col('genre').is_not_null() & (pl.col('genre') != '')).unique(['id','genre']).group_by('genre').agg(pl.len().alias('jumlah_film')).sort('jumlah_film', descending=True)
top_genres = genres.head(15).sort('jumlah_film')
fig4 = px.bar(top_genres.to_dict(as_series=False), x='jumlah_film', y='genre', orientation='h', title='4. Lima belas genre terbanyak, 2000–2023', labels={'jumlah_film':'Jumlah film unik','genre':'Genre'})
save_show(fig4, '04_komposisi_genre')
genres.write_csv(TABLES / 'genre.csv')


C:\Users\DELLA\OneDrive\Documents\ANALISIS BIG DATA\tugas1-abd-2026-main\tugas1-abd-2026-main\notebooks\03_eda_and_insights.ipynb:1: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  {


### Visualisasi 5: Pola bulanan
Heatmap menunjukkan persentase film per bulan di masing-masing tahun, sehingga tahun besar tidak otomatis mendominasi. Tanggal 1 Januari bisa merupakan tanggal perkiraan dalam katalog; grafik ini tidak membuktikan musim permintaan penonton.

In [7]:
monthly = df.group_by('tahun','bulan').agg(pl.len().alias('jumlah_film'))
grid = pl.DataFrame({'tahun':list(range(START, END+1))}).join(pl.DataFrame({'bulan':list(range(1,13))}), how='cross')
monthly = grid.join(monthly, on=['tahun','bulan'], how='left').with_columns(pl.col('jumlah_film').fill_null(0)).with_columns((100*pl.col('jumlah_film')/pl.col('jumlah_film').sum().over('tahun')).alias('persen')).sort('tahun','bulan')
z = monthly['persen'].to_numpy().reshape(END-START+1,12)
fig5 = px.imshow(z, x=['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des'], y=list(range(START,END+1)), aspect='auto', color_continuous_scale='Blues', labels={'x':'Bulan rilis','y':'Tahun','color':'% per tahun'}, title='5. Komposisi bulan rilis dalam setiap tahun')
save_show(fig5, '05_pola_bulanan')
monthly.write_csv(TABLES / 'pola_bulanan.csv')


### Visualisasi 6: Korelasi multidimensi
Anggaran dan pendapatan berskala log; warna menunjukkan rating dan ukuran menunjukkan jumlah vote. Semua film dalam kohort finansial ditampilkan. Pendapatan adalah nilai nominal yang belum disesuaikan inflasi; rasio pendapatan/anggaran bukan laba karena biaya pemasaran, distribusi, dan pembagian pendapatan tidak tercakup.

In [8]:
fig6 = px.scatter(financial.to_dict(as_series=False), x='budget', y='revenue', color='vote_average', size='vote_count', size_max=32, hover_name='title', hover_data=['tahun','rasio_pendapatan_anggaran'], log_x=True, log_y=True, color_continuous_scale='Viridis', title=f'6. Anggaran, pendapatan, rating, dan vote (n={financial.height:,})', labels={'budget':'Anggaran nominal (USD)','revenue':'Pendapatan nominal (USD)','vote_average':'Rating','vote_count':'Jumlah vote'})
save_show(fig6, '06_finansial_multidimensi')
assert len(figures) == 6


## Bagian 4: Rumusan 5 Insight Analitik Utama
Angka berikut dihitung dari data aktual setiap kali notebook dijalankan; rekomendasi bersifat eksploratif.

In [9]:
peak = annual.sort('jumlah_film', descending=True).row(0, named=True)
first, last = annual.row(0, named=True), annual.row(-1, named=True)
q = rated.select(pl.col('vote_average').quantile(.25).alias('q1'), pl.col('vote_average').median().alias('median'), pl.col('vote_average').quantile(.75).alias('q3')).row(0,named=True)
r = corr['log10 pendapatan'][0]
g = genres.row(0,named=True)
month_totals = monthly.group_by('bulan').agg(pl.col('jumlah_film').sum()).sort('jumlah_film',descending=True)
m = month_totals.row(0,named=True)
jan1 = df.filter((pl.col('bulan') == 1) & (pl.col('tanggal').dt.day() == 1)).height
coverage = 100*financial.height/df.height
text = f"""1. **Tren waktu dan musiman:** jumlah film tercatat meningkat dari {first['jumlah_film']:,} pada {START} menjadi {last['jumlah_film']:,} pada {END}. Tahun dengan jumlah tertinggi adalah {peak['tahun']} ({peak['jumlah_film']:,} film). Bulan {m['bulan']} memiliki jumlah rilis terbesar secara agregat ({m['jumlah_film']:,}). Ini adalah pola cakupan katalog, bukan bukti pertumbuhan produksi atau permintaan; ada {jan1:,} tanggal 1 Januari yang berpotensi memuat tanggal perkiraan.

2. **Distribusi dan anomali:** pada {rated.height:,} film dengan minimal {MIN_VOTES} vote, median rating {q['median']:.2f}, dengan 50% bagian tengah antara {q['q1']:.2f}–{q['q3']:.2f}. Batas vote membantu menghindari kesimpulan berdasarkan sangat sedikit penilai, tetapi hasil hanya mewakili film yang cukup banyak dinilai.

3. **Korelasi kunci:** korelasi Pearson antara log10 anggaran dan log10 pendapatan sebesar {r:.3f} pada {corr_data.height:,} film dengan runtime positif dan data finansial tersedia. Hubungan ini dapat menjadi dasar segmentasi eksploratif, tetapi tidak membuktikan kenaikan anggaran menyebabkan kenaikan pendapatan; promosi dan distribusi belum dikendalikan.

4. **Perilaku kategori:** genre {g['genre']} paling sering tercatat, sebanyak {g['jumlah_film']:,} film ({100*g['jumlah_film']/df.height:.2f}% kohort). Karena label genre tumpang tindih, dominasi jumlah judul tidak sama dengan pangsa penonton atau pendapatan.

5. **Rekomendasi praktis:** gunakan segmentasi genre dan tahun sebelum membandingkan performa finansial. Hanya {financial.height:,} film ({coverage:.2f}% kohort) memenuhi syarat finansial dan vote, sehingga keputusan pendanaan tidak boleh digeneralisasi ke seluruh katalog. Tambahkan biaya distribusi, koreksi inflasi, dan audit tanggal 1 Januari sebelum membuat rekomendasi investasi atau penjadwalan rilis.
"""
display(Markdown(text))
(TABLES.parent / 'milestone3_insights.md').write_text('# Insight Milestone 3\n\n'+text, encoding='utf-8')


1. **Tren waktu dan musiman:** jumlah film tercatat meningkat dari 8,735 pada 2000 menjadi 42,709 pada 2023. Tahun dengan jumlah tertinggi adalah 2022 (45,145 film). Bulan 1 memiliki jumlah rilis terbesar secara agregat (102,340). Ini adalah pola cakupan katalog, bukan bukti pertumbuhan produksi atau permintaan; ada 66,772 tanggal 1 Januari yang berpotensi memuat tanggal perkiraan.

2. **Distribusi dan anomali:** pada 12,677 film dengan minimal 100 vote, median rating 6.46, dengan 50% bagian tengah antara 5.88–7.03. Batas vote membantu menghindari kesimpulan berdasarkan sangat sedikit penilai, tetapi hasil hanya mewakili film yang cukup banyak dinilai.

3. **Korelasi kunci:** korelasi Pearson antara log10 anggaran dan log10 pendapatan sebesar 0.675 pada 4,689 film dengan runtime positif dan data finansial tersedia. Hubungan ini dapat menjadi dasar segmentasi eksploratif, tetapi tidak membuktikan kenaikan anggaran menyebabkan kenaikan pendapatan; promosi dan distribusi belum dikendalikan.

4. **Perilaku kategori:** genre Drama paling sering tercatat, sebanyak 123,174 film (20.04% kohort). Karena label genre tumpang tindih, dominasi jumlah judul tidak sama dengan pangsa penonton atau pendapatan.

5. **Rekomendasi praktis:** gunakan segmentasi genre dan tahun sebelum membandingkan performa finansial. Hanya 4,689 film (0.76% kohort) memenuhi syarat finansial dan vote, sehingga keputusan pendanaan tidak boleh digeneralisasi ke seluruh katalog. Tambahkan biaya distribusi, koreksi inflasi, dan audit tanggal 1 Januari sebelum membuat rekomendasi investasi atau penjadwalan rilis.


### Kesimpulan dan reproduktivitas
Notebook memenuhi operasi temporal melalui `group_by_dynamic`, `rolling_mean`, dan DuckDB `LAG OVER`; enam grafik diekspor sebagai HTML mandiri di `output/figures/` (bisa dibuka tanpa internet). Tabel agregat dan audit tersimpan di `output/tables/`. Data mentah dan hasil cleaning Milestone 2 tidak diubah.

**AI Disclosure:** OpenAI Codex membantu menyusun kode EDA, visualisasi, penjelasan, dan menjalankan notebook pada dataset lokal. Mahasiswa tetap perlu membaca setiap sel, memeriksa definisi kohort dan hasil, melengkapi identitas, serta mampu menjelaskan analisis saat pengumpulan.
